In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# read the bronze tabel
df = spark.table("workspace.bronze.flights_raw")

# clean the spaces in columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        col_ref = F.col(f"`{field.name}`")

        df = df.withColumn(
            field.name,
            F.when(
                F.trim(col_ref) == "",
                F.lit(None).cast("string")
            ).otherwise(F.trim(col_ref))
        )

# save this modifided version into silver 
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("workspace.silver.flights_clean")
)

print("Silver table created successfully.")


Silver table created successfully.
